In [1]:
import pandas as pd

In [2]:
pub_author = "../../data/dropbox/LaborDivision/data/cleanedraw/pub_author.pkl"

In [3]:
pub_author = pd.read_pickle(pub_author)

In [6]:
pub_author.head()

,ost_bk,seqno,cluster_id,lastname,cntry
121326103,29768068,1.0,40806321.0,Miron,CAN
39225577,29768068,2.0,41639269.0,Hall,CAN
94571783,29768068,5.0,53722752.0,Antel,CAN
14875820,29768070,1.0,65683243.0,Shen,JPN
88471668,29768070,2.0,79129017.0,Tanida,JPN


In [4]:
pub_author[pub_author.lastname.isna()]

,ost_bk,seqno,cluster_id,lastname,cntry


In [9]:
def return_alpha(df):
    """return the alphabetically type for each paper
    """
    
    df = df.sort_values(by=['ost_bk','seqno'])
    df = df.groupby(['ost_bk']).agg({'lastname':list}).reset_index()
    
    def detect_alpha(namelist):
        if len(namelist)==1:
            return "single"
        elif all(namelist[i]<=namelist[i+1] for i in range(len(namelist)-1)):
            return "alpha"
        else:
            return "noalp"
    
    df['alpha'] = df['lastname'].apply(lambda x:detect_alpha(x))
    
    return df[['ost_bk','alpha']]

In [10]:
alpha_df = return_alpha(pub_author)

In [16]:
pub_author.head()

,ost_bk,seqno,cluster_id,lastname,cntry
121326103,29768068,1.0,40806321.0,Miron,CAN
39225577,29768068,2.0,41639269.0,Hall,CAN
94571783,29768068,5.0,53722752.0,Antel,CAN
14875820,29768070,1.0,65683243.0,Shen,JPN
88471668,29768070,2.0,79129017.0,Tanida,JPN


In [17]:
def return_intl(df):
    """return whether the publication is internationally coauthored
    """
    df = df.groupby(['ost_bk']).agg({'cntry':list}).reset_index()
    df['intl'] = df['cntry'].apply(lambda x: 1 if len(set(x))>1 else 0)
    
    return df[['ost_bk','intl']]

In [18]:
intl_df = return_intl(pub_author)

In [22]:
def return_role(df):
    df['num_author'] = df.groupby(['ost_bk'])['seqno'].transform('max')
    
    def assign_role(seqno, num_author):
        if seqno==1:
            return "F"
        elif seqno==num_author:
            return "L"
        else:
            return "M"
        
    df['role'] = df.apply(lambda x:assign_role(x.seqno, x.num_author),axis=1)
    return df

In [23]:
pub_author = return_role(pub_author)

In [42]:
pub_info = pd.read_pickle("../../data/dropbox/LaborDivision/data/cleanedraw/pub.pkl")

In [44]:
pub_info.head(2)

,ost_bk,year,dis,title
0,69470021,2018,18,From early stress to 12-month development in v...
1,69470022,2020,77,Evolution of robust high speed optical-flow-ba...


In [48]:
pub_info.loc[:,'ost_bk']=pub_info['ost_bk'].astype('int64')

In [49]:
pub_info_meta = pub_info.merge(alpha_df,on='ost_bk')
pub_info_meta = pub_info_meta.merge(intl_df,on='ost_bk')

In [50]:
pub_info_meta.head()

,ost_bk,year,dis,title,alpha,intl
0,69470021,2018,18,From early stress to 12-month development in v...,noalp,0
1,69470022,2020,77,Evolution of robust high speed optical-flow-ba...,alpha,0
2,69470023,2015,41,Activation of the GLP-1 Receptor by Liraglutid...,noalp,0
3,69470024,2013,58,A 7-Year Case of Furosemide-Induced Immune Thr...,noalp,0
4,69470046,2012,77,Study of semiconducting nanomaterials under pr...,alpha,0


In [90]:
pub_author = pub_author.drop(columns=['seqno','lastname','num_author'])

In [91]:
pub_author = pub_author.merge(pub_info_meta[['ost_bk','year','dis']],on='ost_bk')

In [92]:
pub_author.head()

,ost_bk,cluster_id,cntry,role,year,dis
0,29768068,40806321.0,CAN,F,2008,56
1,29768068,41639269.0,CAN,M,2008,56
2,29768068,53722752.0,CAN,L,2008,56
3,29768070,65683243.0,JPN,F,2008,51
4,29768070,79129017.0,JPN,M,2008,51


In [94]:
def select_intl_alpha(intl_signal, alpha_signal, pub_info, pub_author):
    selected_ids = pub_info[(pub_info.intl==intl_signal)&(pub_info.alpha.isin(alpha_signal))]['ost_bk'].unique()
    selected_df = pub_author[pub_author.ost_bk.isin(selected_ids)]
    return selected_df

In [95]:
selected_intl = select_intl_alpha(1,['single','alpha','noalp'],pub_info_meta,pub_author)

In [108]:
selected_intl_noalp = select_intl_alpha(1,['noalp'],pub_info_meta,pub_author)

In [110]:
selected_intl.to_csv("../../data/dropbox/LaborDivision/data/deriveddata/international/pub_author.csv",index=False)

In [111]:
selected_intl_noalp.to_csv("../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/pub_author.csv",index=False)

In [112]:
pub_author.shape

(82155046, 6)

In [113]:
pub_info_meta.shape

(17072488, 6)

In [114]:
pub_author.to_csv("../../data/dropbox/LaborDivision/data/deriveddata/all/pub_author.csv",index=False)

In [115]:
pub_info_meta.to_csv("../../data/dropbox/LaborDivision/data/deriveddata/all/pub_info.csv",index=False)

In [ ]:
len(pub_author.ost_bk.unique())

In [2]:
pub_all = pd.read_csv("../../data/dropbox/LaborDivision/data/deriveddata/all/pub_author.csv")

In [4]:
len(pub_all['ost_bk'].unique())

17072488

In [5]:
17072488/20211823

0.8446782855757247

In [7]:
len(pub_all[pub_all.intl==1]['ost_bk'].unique())

3661174

In [8]:
3661174/17072488

0.2144487669284055

In [10]:
len(pub_all[(pub_all.alpha=='noalp')&(pub_all.intl==1)]['ost_bk'].unique())

3195989

In [11]:
3195989/3661174

0.8729410292982525